In [1]:
import pandas as pd

In [2]:
df_values = pd.read_csv(r"..\main\mean_monthlyvalues.csv")
df_LEZ_area = pd.read_csv(r"..\main\Visualisation\LEZ_area.csv")

In [3]:
df_values.head()

,Year_Month,Province,Average NO2 Value,Average PM2.5 Value,Average PM10 Value
0,1990-01,Drenthe,29,NaN,NaN
1,1990-01,Flevoland,33,NaN,NaN
2,1990-01,Friesland,28,NaN,NaN
3,1990-01,Gelderland,39,NaN,NaN
4,1990-01,Groningen,28,NaN,NaN


In [4]:
df_LEZ_area.head()

,Unnamed: 0,zone_name,start_date,end_date,area_km2,province
0,0,LEZ 's-Hertogenbosch,2020-01-01T00:00:00Z,2025-03-01T00:00:00Z,2.856116,North-Brabant
1,1,LEZ Delft,2020-01-01T00:00:00Z,2999-01-01T00:00:00Z,1.374783,Zuid-Holland
2,2,LEZ Haarlem,2020-01-01T00:00:00Z,2999-01-01T00:00:00Z,11.509328,Noord-Holland
3,3,LEZ Apeldoorn,2025-01-01T00:00:00Z,2026-12-31T23:00:00Z,1.929479,Gelderland
4,4,LEZ Den Haag,2020-01-01T00:00:00Z,2999-01-01T00:00:00Z,12.110339,Zuid-Holland


In [5]:
df_values['Year_Month'] = pd.to_datetime(df_values['Year_Month'], format='%Y-%m')

# Convert start_date and end_date to datetime
df_LEZ_area['start_date'] = pd.to_datetime(df_LEZ_area['start_date'], errors='coerce')
df_LEZ_area['end_date'] = pd.to_datetime(df_LEZ_area['end_date'], errors='coerce')

# Normalize to month start to match Year_Month convention
df_LEZ_area['start_date'] = df_LEZ_area['start_date'].dt.to_period('M').dt.to_timestamp()
df_LEZ_area['end_date'] = df_LEZ_area['end_date'].dt.to_period('M').dt.to_timestamp()

df_LEZ_area['end_date'] = df_LEZ_area['end_date'].fillna(pd.Timestamp('2100-12-01'))

C:\Users\Utente\AppData\Local\Temp\ipykernel_21092\3321452805.py:8: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df_LEZ_area['start_date'] = df_LEZ_area['start_date'].dt.to_period('M').dt.to_timestamp()
C:\Users\Utente\AppData\Local\Temp\ipykernel_21092\3321452805.py:9: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df_LEZ_area['end_date'] = df_LEZ_area['end_date'].dt.to_period('M').dt.to_timestamp()


In [6]:
df_LEZ_area.head()

,Unnamed: 0,zone_name,start_date,end_date,area_km2,province
0,0,LEZ 's-Hertogenbosch,2020-01-01,2025-03-01,2.856116,North-Brabant
1,1,LEZ Delft,2020-01-01,2100-12-01,1.374783,Zuid-Holland
2,2,LEZ Haarlem,2020-01-01,2100-12-01,11.509328,Noord-Holland
3,3,LEZ Apeldoorn,2025-01-01,2026-12-01,1.929479,Gelderland
4,4,LEZ Den Haag,2020-01-01,2100-12-01,12.110339,Zuid-Holland


In [7]:
def get_area(row):
    province = row['Province']
    date = row['Year_Month']
    # Filter by province
    matches = df_LEZ_area[
        (df_LEZ_area['province'] == province) &
        (df_LEZ_area['start_date'] <= date) &
        (df_LEZ_area['end_date'] >= date)
    ]

    # Sum all matching areas (if multiple)
    area_sum = matches['area_km2'].sum() if not matches.empty else 0

    # Add to existing value if present, else just area_sum
    current_val = row['area_km2'] if pd.notna(row.get('area_km2', None)) else 0
    return current_val + area_sum

# Initiate area_km2 column if doesn't exist
if 'area_km2' not in df_values.columns:
    df_values['area_km2'] = 0

df_values['area_km2'] = df_values.apply(get_area, axis=1)

In [8]:
display(df_values)

,Year_Month,Province,Average NO2 Value,Average PM2.5 Value,Average PM10 Value,area_km2
0,1990-01-01,Drenthe,29,NaN,NaN,0.000000
1,1990-01-01,Flevoland,33,NaN,NaN,0.000000
2,1990-01-01,Friesland,28,NaN,NaN,0.000000
3,1990-01-01,Gelderland,39,NaN,NaN,0.000000
4,1990-01-01,Groningen,28,NaN,NaN,0.000000
...,...,...,...,...,...,...
1435,2024-12-01,Noord-Holland,20,10.0,16.0,114.428586
1436,2024-12-01,Overijssel,9,6.0,11.0,0.000000
1437,2024-12-01,Utrecht,16,11.0,14.0,5.308662
1438,2024-12-01,Zeeland,10,NaN,16.0,0.000000


In [9]:
df_values.to_csv('df_values.csv')